# Step 1 Phase A-11：CMBtopology の x₀ 規約と A6 観測者規約の橋渡し（数値検証）**v1.3.1**
2026-09-07。共分散生成の BLOCKER。v1.3 監査（SCIENCE DESIGN: APPROVED）の極小 patch：**smoke provenance chain gate**（official が lock に記録された
smoke provenance の実在・SHA・status・全 gate・canonical notebook SHA・environment fingerprint を再計算・照合）／変位励起の早期 assert／gate 名の整理。
v1.3 の formal hardening：`t2b2_run.py` の gate／live dependency 版の gate／SMOKE_PASS と OFFICIAL の分離／
env lock を成功した smoke の最後に書き official でその status を要求／pip freeze は診断に降格（直接依存 10 package＋platform/BLAS を exact lock）／case 名の exact set／変位成分の励起 gate ほか。

## v1.3 までの変更
1. **変位比較を真の二経路に**：A6 側は `results/step1_phaseA/A6A7_freeze/s1_phaseA6A7_v1.4.2.py`（SHA gate）から `family_data()` を
   AST 抽出して (M_A6, T_A6, Λ_A6) を得る。CT 側は **pinned CMBtopology ソース**から `M_A/M_B`（literal）・`T_A/T_B`（式を評価）・
   E2 の半回転は位相因子 `2i(k_x x0_x + k_y x0_y)` のパターンから M を導く。両者の M・T の一致と，δ_A6=(M_A6−I)r+T_A6 vs
   δ_CT=(I−M_CT)b+T_CT の ker 平行/垂直成分の一致を独立 gate に。Λ_CT はソースから安全に抽出できないため，
   **格子束縛は全基底ベクトルの並進不変性電池で経験的に検証**（E7 3・E7tilt 3・E8 3・E2 3）。
2. **環境 lock**：`A11_MODE='smoke'` で `a11_env_lock.json`（importlib.metadata による exact 版・requirements SHA・pip freeze）を生成，
   `'official'` は lock を読んで exact 一致を hard gate。**environment fingerprint を cache key と manifest に含める**。
3. **notebook 同一性**：origin/main の同名 notebook の source-only SHA と live source SHA を比較（official では required gate）。
4. **live module 同一性**：`sys.modules` を purge → import → `__file__` と import 後の live file SHA を gate。
5. 全並進基底の不変性・kernel 方向移動（E8 z・E2 z・E7tilt 面内）・exact gate inventory（`set(GATES)==set(REQUIRED)`・診断は別辞書）・
   cache key の完全化（do_polarization/normalize/l_range/requirements SHA/env fingerprint・x0 は full precision）・white metric の guard・
   `not_lattice_degenerate` と `empirically_discriminating` の分離・shallow clone の commit fetch。

## 実行モード
- **smoke**（先に1回・約20分）：E7 base＋H1・E2 base の3共分散で import/出力/形状/intake/cache を確認し，env lock を生成。
- **official**（約4時間・cache 再開可）：35 共分散・24 行。最後に `assert OFFICIAL`。

In [ ]:
# ---- 1. モード・環境・ソース同一性（import 前に SHA 検査・sys.modules purge・import 後に live 検査）----
import os, sys, json, hashlib, subprocess, time, glob, shutil, datetime, importlib, inspect, re, ast
from pathlib import Path
from importlib.metadata import version as pkg_version, PackageNotFoundError
A11_MODE = globals().get('A11_MODE', os.environ.get('A11_MODE', 'official'));  assert A11_MODE in ('smoke', 'official'), A11_MODE
IN_COLAB = os.path.isdir('/content')
WORK = '/content' if IN_COLAB else os.environ.get('A11_WORK', os.path.join(os.getcwd(), 'a11_work'))
BASE = '/content/drive/MyDrive/mirror_topology' if IN_COLAB else os.environ.get('A11_BASE', os.path.join(WORK, 'base'))
if IN_COLAB:
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    assert os.path.isdir('/content/drive/MyDrive'), 'Driveマウント失敗：明示停止'
os.makedirs(WORK, exist_ok=True)
OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a11_v1.3.1_{A11_MODE}'); CACHE = os.path.join(OUT, 'cov_cache'); SCRATCH = os.path.join(WORK, 'ct_scratch', A11_MODE)
LOCK_PATH = os.path.join(BASE, 'runs_step1_phaseA', 'a11_env_lock.json')
for d in (OUT, CACHE, SCRATCH): os.makedirs(d, exist_ok=True)
EXPECTED_CMBTOPO_COMMIT = '0cc65e34f03df85e92f738686bff0a476132f337'
EXPECTED_MT_COMMIT = 'efa23136e99be70814d17a4845bcdca0eac1bada'      # A9 freeze commit: contains frozen t1_engine / t2b2_bridge / A6A7_freeze
EXPECTED_SHA = dict(t1_engine='87bf8424073af021264b12fe312ab5255b71008bdd5fe874d164d48daf034dc8', bridge='45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872',
                    run='03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db', a6_script='661ed0b01f35853263d9374f56960dd8f87d7bdf68070f2493d5576e702e2b2e')
NB_BASENAME = 'MirrorTopology_Step1_A11_x0bridge_v1.3.1.ipynb'
GATES = {}; DIAG = {}; GIT_LOG = []
def sha(p):
    with open(p, 'rb') as fh: return hashlib.sha256(fh.read()).hexdigest()
def run(c, **kw):
    r = subprocess.run(c, capture_output=True, text=True, check=True, **kw); GIT_LOG.append(dict(cmd=c, stderr=r.stderr.strip()[:300])); return r.stdout.strip()
_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
def pin_repo(url, path, commit, canonical):
    if not os.path.isdir(os.path.join(path, '.git')): run(['git', 'clone', url, path], env=_env)
    if run(['git', '-C', path, 'rev-parse', '--is-shallow-repository']) == 'true': run(['git', '-C', path, 'fetch', '-q', '--unshallow'], env=_env)
    run(['git', '-C', path, 'fetch', '-q', 'origin', commit], env=_env); run(['git', '-C', path, 'reset', '-q', '--hard', commit]); run(['git', '-C', path, 'clean', '-fdx', '-q'])
    return dict(head=run(['git', '-C', path, 'rev-parse', 'HEAD']), origin=run(['git', '-C', path, 'remote', 'get-url', 'origin']), clean=(run(['git', '-C', path, 'status', '--porcelain']) == ''), canonical=canonical)
MT = os.path.join(WORK, 'mt_a11'); mt = pin_repo('https://github.com/tsujikeita/mirror-topology.git', MT, EXPECTED_MT_COMMIT, 'https://github.com/tsujikeita/mirror-topology')
GATES['G_mt_commit'] = (mt['head'] == EXPECTED_MT_COMMIT); GATES['G_mt_origin'] = (mt['origin'].rstrip('/').removesuffix('.git') == mt['canonical']); GATES['G_mt_clean_preimport'] = mt['clean']
A6_SCRIPT = os.path.join(MT, 'results', 'step1_phaseA', 'A6A7_freeze', 's1_phaseA6A7_v1.4.2.py')
GATES['G_t1_engine_sha'] = (sha(os.path.join(MT, 't1_engine.py')) == EXPECTED_SHA['t1_engine']); GATES['G_bridge_sha'] = (sha(os.path.join(MT, 't2b2_bridge.py')) == EXPECTED_SHA['bridge'])
GATES['G_t2b2_run_sha'] = (sha(os.path.join(MT, 't2b2_run.py')) == EXPECTED_SHA['run'])
GATES['G_a6_artifact_sha'] = (sha(A6_SCRIPT) == EXPECTED_SHA['a6_script'])
assert all(GATES.values()), GATES
CT_DIR = os.path.join(WORK, 'CMBtopology_pinned'); ct = pin_repo('https://github.com/CompactCollaboration/CMBtopology.git', CT_DIR, EXPECTED_CMBTOPO_COMMIT, 'https://github.com/CompactCollaboration/CMBtopology')
GATES['G_ct_commit'] = (ct['head'] == EXPECTED_CMBTOPO_COMMIT); GATES['G_ct_origin'] = (ct['origin'].rstrip('/').removesuffix('.git') == ct['canonical']); GATES['G_ct_clean_preimport_incl_untracked'] = ct['clean']
CT_SRC_SHA = {f: sha(os.path.join(CT_DIR, 'topology', p)) for f, p in [('E2.py', 'src/E2.py'), ('E7.py', 'src/E7.py'), ('E8.py', 'src/E8.py'), ('run_topology.py', 'run_topology.py')]}
assert all(GATES.values()), GATES
REQ_TXT = os.path.join(CT_DIR, 'requirements.txt'); REQ_SHA = sha(REQ_TXT)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'healpy==1.20.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ_TXT], check=True)
# ---- environment lock ----
PKGS = ['numpy', 'scipy', 'matplotlib', 'healpy', 'camb', 'tqdm', 'numba', 'quaternionic', 'spherical', 'pandas']
def pkgver(n):
    try: return pkg_version(n)
    except PackageNotFoundError: return 'MISSING'
VERS = {n: pkgver(n) for n in PKGS}; PYV = sys.version.split()[0]
GATES['G_ct_dependencies_present'] = all(v != 'MISSING' for v in VERS.values()); assert GATES['G_ct_dependencies_present'], VERS
PIP_FREEZE = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True).stdout
PIP_FREEZE_DIAG_SHA = hashlib.sha256(PIP_FREEZE.encode()).hexdigest()   # diagnostic only (Colab-wide freeze is too volatile for a formal lock)
import platform
PLATFORM = dict(machine=platform.machine(), platform=platform.platform())
LOCK_CANDIDATE = dict(schema='a11_env_lock_v2', python=PYV, versions=None, requirements_sha256=REQ_SHA, platform=PLATFORM, blas_lapack=None,
                      full_pip_freeze_diagnostic_sha256=PIP_FREEZE_DIAG_SHA, cmbtopology_commit=EXPECTED_CMBTOPO_COMMIT, smoke_status='PENDING')
open(os.path.join(OUT, 'pip_freeze.txt'), 'w').write(PIP_FREEZE)
# ---- purge stale modules, import, gate live identity ----
for name in list(sys.modules):
    if name in {'t1_engine', 't2b2_bridge', 't2b2_run', 'topology'} or name.startswith('topology.'): del sys.modules[name]
importlib.invalidate_caches()
sys.path.insert(0, MT); import t1_engine as t1, t2b2_bridge as br, t2b2_run as tr
sys.path.insert(0, CT_DIR); from topology.run_topology import run_topology
import numpy as np, pandas as pd, healpy as hp, scipy, camb, matplotlib, tqdm, numba, quaternionic, spherical
from scipy.special import sph_harm_y
GATES['G_live_import_paths'] = (Path(t1.__file__).resolve() == Path(MT, 't1_engine.py').resolve() and Path(br.__file__).resolve() == Path(MT, 't2b2_bridge.py').resolve()
                                and Path(tr.__file__).resolve() == Path(MT, 't2b2_run.py').resolve() and Path(inspect.getmodule(run_topology).__file__).resolve().is_relative_to(Path(CT_DIR).resolve()))
GATES['G_live_module_sha'] = (sha(t1.__file__) == EXPECTED_SHA['t1_engine'] and sha(br.__file__) == EXPECTED_SHA['bridge'] and sha(tr.__file__) == EXPECTED_SHA['run'])
assert GATES['G_live_import_paths'] and GATES['G_live_module_sha']
LIVE_MODULES = dict(numpy=np, scipy=scipy, matplotlib=matplotlib, healpy=hp, camb=camb, tqdm=tqdm, numba=numba, quaternionic=quaternionic, spherical=spherical, pandas=pd)
LIVE_VERS = {n: str(getattr(m, '__version__', 'MISSING_VERSION_ATTRIBUTE')) for n, m in LIVE_MODULES.items()}
GATES['G_live_dependency_versions'] = (LIVE_VERS == {k: str(v) for k, v in VERS.items()})
assert GATES['G_live_dependency_versions'], ('runtime restart required; do not create smoke lock', {k: (VERS[k], LIVE_VERS[k]) for k in VERS if str(VERS[k]) != LIVE_VERS[k]})
BLAS = str({k: v.get('name') for k, v in np.show_config(mode='dicts').get('Build Dependencies', {}).items() if k in ('blas', 'lapack')})
LOCK_CANDIDATE['versions'] = LIVE_VERS; LOCK_CANDIDATE['blas_lapack'] = BLAS
if A11_MODE == 'official':
    assert os.path.exists(LOCK_PATH), f'official run requires the environment lock from a SMOKE_PASS run: {LOCK_PATH}'
    lock = json.load(open(LOCK_PATH)); LOCK_SHA = sha(LOCK_PATH)
    GATES['G_env_lock'] = (lock.get('schema') == 'a11_env_lock_v2' and lock['python'] == PYV and lock['versions'] == LIVE_VERS and lock['requirements_sha256'] == REQ_SHA
                           and lock['platform'] == PLATFORM and lock['blas_lapack'] == BLAS and lock['cmbtopology_commit'] == EXPECTED_CMBTOPO_COMMIT and lock.get('smoke_status') == 'SMOKE_PASS')
    if not GATES['G_env_lock']: print('lock mismatch:', {k: (lock.get(k), v) for k, v in dict(python=PYV, versions=LIVE_VERS, requirements_sha256=REQ_SHA, platform=PLATFORM, blas_lapack=BLAS, smoke_status='SMOKE_PASS').items() if lock.get(k) != v})
    assert GATES['G_env_lock'], 'environment lock mismatch or lock not from a SMOKE_PASS run: STOP'
else:
    lock = None; LOCK_SHA = None; GATES['G_env_lock_candidate_complete'] = bool(LOCK_CANDIDATE['versions'] and LOCK_CANDIDATE['blas_lapack'])   # smoke: lock written at the END if SMOKE_PASS
ENV_FINGERPRINT = hashlib.sha256(json.dumps(dict(python=PYV, versions=LIVE_VERS, requirements_sha256=REQ_SHA, platform=PLATFORM, blas_lapack=BLAS, ct_commit=EXPECTED_CMBTOPO_COMMIT,
                                                run_config=dict(l_max=4, do_polarization=False, normalize=False, l_range=[[2, 4]], lp_range=[[2, 4]])), sort_keys=True).encode()).hexdigest()
LMAX = 4
# ---- notebook identity: origin/main copy vs live source ----
run(['git', '-C', MT, 'fetch', '-q', 'origin', 'main'], env=_env); MAIN_HEAD = run(['git', '-C', MT, 'rev-parse', 'origin/main'])
try:
    NB_HEAD_SHA = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except subprocess.CalledProcessError: NB_HEAD_SHA = None
NB_LIVE_SHA = tr.live_notebook_source_sha()
if A11_MODE == 'official': GATES['G_notebook_live_source'] = bool(isinstance(NB_LIVE_SHA, str) and NB_HEAD_SHA is not None and NB_LIVE_SHA == NB_HEAD_SHA)
else: GATES['G_notebook_head_available'] = (NB_HEAD_SHA is not None)
DIAG['notebook'] = dict(live=NB_LIVE_SHA, origin_main_head=MAIN_HEAD, head_copy=NB_HEAD_SHA)
if A11_MODE == 'official': assert GATES['G_notebook_live_source'], DIAG['notebook']
if A11_MODE == 'official':
    chain = dict(ok=False)
    try:
        spath = lock.get('smoke_provenance_path'); path_ok = isinstance(spath, str) and os.path.isfile(spath)
        sha_ok = bool(path_ok and sha(spath) == lock.get('smoke_provenance_sha256')); sp = json.load(open(spath)) if sha_ok else {}
        req = sp.get('required_gates')
        status_ok = (sp.get('status') == 'SMOKE_PASS' and sp.get('SMOKE_PASS') is True and sp.get('OFFICIAL') is False)
        gates_ok = (isinstance(req, list) and len(req) > 0 and sp.get('gate_inventory_exact') is True and all(sp.get('gates', {}).get(k) is True for k in req))
        source_ok = (lock.get('canonical_notebook_head_sha') == NB_HEAD_SHA and sp.get('notebook_identity', {}).get('head_copy') == NB_HEAD_SHA)
        environment_ok = (sp.get('environment', {}).get('fingerprint') == ENV_FINGERPRINT)
        chain.update(path=spath, path_ok=path_ok, sha_ok=sha_ok, status_ok=status_ok, gates_ok=gates_ok, source_ok=source_ok, environment_ok=environment_ok)
        chain['ok'] = bool(path_ok and sha_ok and status_ok and gates_ok and source_ok and environment_ok)
    except Exception as e: chain['error'] = f'{type(e).__name__}: {e}'
    DIAG['smoke_provenance_chain'] = chain; GATES['G_smoke_provenance_chain'] = bool(chain['ok'])
    assert GATES['G_smoke_provenance_chain'], chain
print(f'mode={A11_MODE} / mirror-topology @ {mt["head"][:12]} / CMBtopology @ {ct["head"][:12]} / env fingerprint {ENV_FINGERPRINT[:12]}'); print('live versions:', LIVE_VERS); print('notebook:', DIAG['notebook'])

In [ ]:
# ---- 2. D(M)（A9 と同一の求積構成）＋ A11 で使う全 M の gate ----
LM = br.lm_full(); M21 = br.M_matrix()[0]; RB = br.real_basis_lm()
_NT = _NP = 2 * LMAX + 2
_xg, _wg = np.polynomial.legendre.leggauss(_NT); _th = np.arccos(_xg); _ph = 2 * np.pi * np.arange(_NP) / _NP
TH, PH = np.meshgrid(_th, _ph, indexing='ij'); WQ = (np.repeat(_wg[:, None], _NP, axis=1) * (2 * np.pi / _NP)).ravel()
DIRS = np.column_stack([np.sin(TH).ravel() * np.cos(PH).ravel(), np.sin(TH).ravel() * np.sin(PH).ravel(), np.cos(TH).ravel()])
def asha(a): return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
def Yc_at(dirs):
    th, ph = hp.vec2ang(dirs); return np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM])
def Ymat(dirs): return (M21.conj() @ Yc_at(dirs)).real.T
YQ = Ymat(DIRS)
def D_of(Mm): return (YQ * WQ[:, None]).T @ Ymat(DIRS @ Mm)
GATES['G_quadrature_orthonormal'] = bool(np.abs((YQ * WQ[:, None]).T @ YQ - np.eye(21)).max() < 1e-12)
rng = np.random.default_rng(20260907); dirs_t = rng.standard_normal((100, 3)); dirs_t /= np.linalg.norm(dirs_t, axis=1, keepdims=True); th_t, ph_t = hp.vec2ang(dirs_t)
E_explicit = np.column_stack([(sph_harm_y(l, m, th_t, ph_t).real if (cs == 'c' and m == 0) else np.sqrt(2) * sph_harm_y(l, m, th_t, ph_t).real if cs == 'c' else np.sqrt(2) * sph_harm_y(l, m, th_t, ph_t).imag) for (l, m, cs) in RB])
GATES['G_real_basis_bridge'] = bool(np.abs(Ymat(dirs_t) - E_explicit).max() < 1e-12)
I3 = np.eye(3); MA = np.diag([1., -1., 1.]); MB = np.diag([-1., 1., 1.]); RZ = np.diag([-1., -1., 1.])
from scipy.spatial.transform import Rotation
Rr1, Rr2 = Rotation.random(rng=rng).as_matrix(), Rotation.random(rng=rng).as_matrix()
worst_orth = worst_geom = 0.0
for Mm in [MA, MB, MA @ MB, RZ, Rr1, Rr2]:
    D = D_of(Mm); worst_orth = max(worst_orth, np.abs(D.T @ D - np.eye(21)).max())
    x = rng.standard_normal(21); a = M21.conj().T @ x
    worst_geom = max(worst_geom, np.abs((Yc_at(dirs_t).T @ (M21.conj().T @ (D @ x))).real - (Yc_at(dirs_t @ Mm).T @ a).real).max() / np.abs((Yc_at(dirs_t @ Mm).T @ a).real).max())
hom = max(np.abs(D_of(MA @ MB) - D_of(MA) @ D_of(MB)).max(), np.abs(D_of(Rr1 @ Rr2) - D_of(Rr1) @ D_of(Rr2)).max(), np.abs(D_of(MA @ Rr1) - D_of(MA) @ D_of(Rr1)).max())
Dy_th = np.diag([(-1.0 if cs == 's' else 1.0) for (l, m, cs) in RB]); Dz_th = np.diag([(-1.0) ** (l + m) for (l, m, cs) in RB])
GATES['G_D_orthogonal_all_used'] = bool(worst_orth < 1e-10); GATES['G_D_direct_geometry_complex_path'] = bool(worst_geom < 1e-10); GATES['G_D_homomorphism'] = bool(hom < 1e-10)
GATES['G_reflection_D_analytic'] = bool(np.abs(D_of(MA) - Dy_th).max() < 1e-12 and np.abs(D_of(np.diag([1., 1., -1.])) - Dz_th).max() < 1e-12)
assert all(GATES[k] for k in ['G_quadrature_orthonormal', 'G_real_basis_bridge', 'G_D_orthogonal_all_used', 'G_D_direct_geometry_complex_path', 'G_D_homomorphism', 'G_reflection_D_analytic']), GATES
DIAG['representation'] = dict(worst_orthogonality=float(worst_orth), worst_direct_geometry=float(worst_geom), worst_homomorphism=float(hom), quadrature_nodes_sha256=asha(DIRS), quadrature_weights_sha256=asha(WQ), M21_sha256=asha(M21),
                              LM_sha256=hashlib.sha256(json.dumps(LM).encode()).hexdigest(), RB_sha256=hashlib.sha256(json.dumps(RB).encode()).hexdigest())
print('D(M) gates OK')

In [ ]:
# ---- 3. 二経路の生成子：A6 凍結 artifact（AST 抽出） vs pinned CMBtopology ソース（抽出） ----
_a6src = open(A6_SCRIPT).read(); _fn = [n for n in ast.parse(_a6src).body if isinstance(n, ast.FunctionDef) and n.name == 'family_data'][0]
_ns = {'np': np, 'I3': I3}; exec(ast.get_source_segment(_a6src, _fn), _ns); family_data_A6 = _ns['family_data']
def gens_A6(top, p):
    fd = family_data_A6(top, 1.0, **{k: v for k, v in p.items() if k not in ('alpha', 'beta', 'gamma')}) if top != 'E2' else family_data_A6('E2', 1.0, Lx=p['Lx'], Ly=p['Ly'], Lz=p['Lz'])
    return {cid: (M, v) for cid, M, v in fd['cosets'] if cid != 'id'}, fd['lattice']
def gens_CT(top, p):
    """CMBtopology side, extracted from the pinned source files (independent of A6)."""
    src = open(os.path.join(CT_DIR, 'topology', 'src', f'{top}.py')).read().replace('\r', '')
    out = {}
    if top in ('E7', 'E8'):
        Ms = {m.group(1): np.array(eval(m.group(2), {'np': np}), float) for m in re.finditer(r'M_([A-Z])\s*=\s*(np\.diag\(\[[^\]]*\]\))', src)}
        Ts = {}
        for m in re.finditer(r'T_([A-Z])\s*=\s*np\.array\(\[([^\]]*)\]\)', src):
            Ts[m.group(1)] = np.array(eval('[' + m.group(2) + ']', {**{k: float(v) for k, v in p.items()}}), float)
        for k in Ms: out[f'glide_{k}'] = (Ms[k], Ts[k])
    else:
        m = re.search(r'T_B\s*=\s*L_z\s*\*\s*np\.array\(\[([^\]]*)\]\)', src); assert m, 'E2 T_B not found'
        from math import cos, sin, pi
        beta, gamma = p['beta'] * pi / 180, p['gamma'] * pi / 180
        T_B = p['Lz'] * np.array(eval('[' + m.group(1) + ']', {'cos': cos, 'sin': sin, 'beta': beta, 'gamma': gamma}), float)
        # half-turn: phase factor exp(2i(k_x x0_x + k_y x0_y)) exp(i k.T_B) = exp(-i k.(M x0 - T_B)) with M = diag(-1,-1,1)
        assert re.search(r'2\*1j\*\(k_x\*x0\[0\]\s*\+\s*k_y\*x0\[1\]\)', src), 'E2 half-turn phase pattern not found'
        out['halfturn_B'] = (np.diag([-1., -1., 1.]), T_B)
    return out
def E7_shape(LAx=1.0, LAy=0.3, L1y=1.0, L2x=0.0, L2z=1.0): return dict(LAx=LAx, LAy=LAy, L1y=L1y, L2x=L2x, L2z=L2z)
def E8_shape(LAx=1.0, LAy=0.3, LBx=0.2, LBz=1.0, LCy=1.0): return dict(LAx=LAx, LAy=LAy, LBx=LBx, LBz=LBz, LCy=LCy)
def E2_shape(Lx=1.0, Ly=1.0, Lz=1.0): return dict(Lx=Lx, Ly=Ly, Lz=Lz, alpha=90.0, beta=90.0, gamma=0.0)
b1 = np.array([0.31, 0.21, 0.42]); b2 = np.array([0.13, 0.44, 0.27]); p7, p7t, p8, p2 = E7_shape(), E7_shape(LAy=0.25, L2x=0.5), E8_shape(), E2_shape()
CASES = []; bind_M = bind_T = 0.0
for cid, top, p, b, g in [('E7_b1_A', 'E7', p7, b1, 'glide_A'), ('E7_b2_A', 'E7', p7, b2, 'glide_A'), ('E7tilt_b1_A', 'E7', p7t, b1, 'glide_A'),
                          ('E8_b1_A', 'E8', p8, b1, 'glide_A'), ('E8_b1_B', 'E8', p8, b1, 'glide_B'), ('E2_b1_B', 'E2', p2, b1, 'halfturn_B')]:
    gA6, LamA6 = gens_A6(top, p); MA6, TA6 = gA6[g]; MCT, TCT = gens_CT(top, p)[g]
    bind_M = max(bind_M, np.abs(MA6 - MCT).max()); bind_T = max(bind_T, np.abs(TA6 - TCT).max())
    r = -b; dA6 = (MA6 - I3) @ r + TA6; dCT = (I3 - MCT) @ b + TCT                 # two independent sources
    U_, S_, Vt = np.linalg.svd(MA6 - I3); ker = Vt[S_ < 1e-9].T; Ppar = ker @ ker.T if ker.size else np.zeros((3, 3)); Pperp = I3 - Ppar
    c2T = np.linalg.solve(LamA6, 2 * TA6); not_deg = not np.allclose(c2T, np.round(c2T), atol=1e-9, rtol=0.0)
    CASES.append(dict(case=cid, topology=top, shape=p, base=b, gen=g, M=MA6, T=TA6, lattice=LamA6, not_lattice_degenerate=not_deg, x0_H1=MCT @ b - TCT, x0_H2=MCT @ b + TCT,
                      dpar_err=float(np.abs(Ppar @ (dA6 - dCT)).max()), dperp_err=float(np.abs(Pperp @ (dA6 - dCT)).max()), dpar_norm=float(np.linalg.norm(Ppar @ dA6)), dperp_norm=float(np.linalg.norm(Pperp @ dA6))))
GATES['G_a6_ct_M_binding'] = bool(bind_M < 1e-12); GATES['G_a6_ct_T_binding'] = bool(bind_T < 1e-12)
GATES['G_displacement_parallel_independent'] = bool(all(c['dpar_err'] < 1e-12 for c in CASES)); GATES['G_displacement_perpendicular_independent'] = bool(all(c['dperp_err'] < 1e-12 for c in CASES))
GATES['G_displacement_components_excited'] = bool(all(c['dpar_norm'] > 1e-6 and c['dperp_norm'] > 1e-6 for c in CASES))
assert all(GATES[k] for k in ['G_a6_ct_M_binding', 'G_a6_ct_T_binding', 'G_displacement_parallel_independent', 'G_displacement_perpendicular_independent', 'G_displacement_components_excited']), (bind_M, bind_T)
def lat_of(top, p): return gens_A6(top, p)[1]
INV = []
for top, p, nm in [('E7', p7, 'E7'), ('E7', p7t, 'E7tilt'), ('E8', p8, 'E8'), ('E2', p2, 'E2')]:
    L = lat_of(top, p)
    for j in range(3): INV.append(dict(case=f'{nm}_b1_lat{j}', topology=top, shape=p, base=b1, x0=b1 + L[:, j], kind=f'{nm}_lattice_translation'))
INV += [dict(case='E7_b1_xshift', topology='E7', shape=p7, base=b1, x0=b1 + np.array([0.2, 0., 0.]), kind='E7_invariant_plane_shift'),
        dict(case='E7_b1_zshift', topology='E7', shape=p7, base=b1, x0=b1 + np.array([0., 0., 0.15]), kind='E7_invariant_plane_shift'),
        dict(case='E7tilt_b1_xshift', topology='E7', shape=p7t, base=b1, x0=b1 + np.array([0.2, 0., 0.]), kind='E7tilt_invariant_plane_shift'),
        dict(case='E8_b1_zshift', topology='E8', shape=p8, base=b1, x0=b1 + np.array([0., 0., 0.15]), kind='E8_kernel_shift'),
        dict(case='E2_b1_zshift', topology='E2', shape=p2, base=b1, x0=b1 + np.array([0., 0., 0.15]), kind='E2_kernel_shift'),
        dict(case='E7_b1_yshift', topology='E7', shape=p7, base=b1, x0=b1 + np.array([0., 0.12, 0.]), kind='y_dependence_control')]
EXPECTED = dict(disc={'E7_b1_A', 'E7_b2_A', 'E7tilt_b1_A', 'E8_b1_A', 'E8_b1_B'}, nond={'E2_b1_B'}, lat_cases={f'{nm}_b1_lat{j}' for nm in ['E7', 'E7tilt', 'E8', 'E2'] for j in range(3)},
                kernel_cases={'E7_b1_xshift', 'E7_b1_zshift', 'E7tilt_b1_xshift', 'E8_b1_zshift', 'E2_b1_zshift'}, ctrl={'E7_b1_yshift'}, n_rows=24, n_unique_cov=35)
if A11_MODE == 'smoke':
    CASES = [c for c in CASES if c['case'] == 'E7_b1_A']; INV = []; EXPECTED = dict(disc={'E7_b1_A'}, nond=set(), lat_cases=set(), kernel_cases=set(), ctrl=set(), n_rows=1, n_unique_cov=3)
print('binding: M', f'{bind_M:.1e}', 'T', f'{bind_T:.1e}', '| displacement gates:', GATES['G_displacement_parallel_independent'], GATES['G_displacement_perpendicular_independent'])
for c in CASES: print(f"  {c['case']:12s} not_lattice_degenerate={c['not_lattice_degenerate']} x0_H1={np.round(c['x0_H1'],3).tolist()} x0_H2={np.round(c['x0_H2'],3).tolist()}")

In [ ]:
# ---- 4. 共分散生成：tag 専用 scratch・atomic cache（env fingerprint 込み）・orphan 回復・全 intake meta ----
RUN_CFG = dict(l_max=LMAX, do_polarization=False, normalize=False, l_range=[[2, LMAX]], lp_range=[[2, LMAX]])
def key_of(top, p, x0): return dict(topology=top, params={k: float(v) for k, v in p.items()}, x0=[float(v) for v in x0], run_config=RUN_CFG, cmbtopology_commit=EXPECTED_CMBTOPO_COMMIT, requirements_sha256=REQ_SHA, env_fingerprint=ENV_FINGERPRINT)
def tag_of(top, p, x0): return top + '_' + hashlib.sha256(json.dumps(key_of(top, p, x0), sort_keys=True).encode()).hexdigest()
def atomic_write_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as fh: json.dump(obj, fh, indent=1); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
def ensure_cov(top, p, x0):
    tag = tag_of(top, p, x0); f = os.path.join(CACHE, f'cov_{tag[:20]}.npy'); mf = f + '.manifest.json'
    if os.path.exists(f) != os.path.exists(mf):
        for q in (f, mf):
            if os.path.exists(q): os.remove(q)
    if os.path.exists(f):
        rec = json.load(open(mf)); assert rec['manifest'] == key_of(top, p, x0) and rec['tag'] == tag, tag; assert sha(f) == rec['cov_file_sha256'], f'再利用SHA不一致: {tag}'; return f, tag, rec
    scratch = os.path.join(SCRATCH, tag[:20]); shutil.rmtree(scratch, ignore_errors=True); os.makedirs(scratch); cwd0 = os.getcwd(); t0 = time.time()
    try:
        os.chdir(scratch)
        run_topology(topology=top, l_max=LMAX, do_polarization=False, normalize=False, l_range=np.array([[2, LMAX]]), lp_range=np.array([[2, LMAX]]), x0=np.array(x0, float), **p)
        dirs = glob.glob('runs/*'); assert len(dirs) == 1, dirs; d = dirs[0]
        src = os.path.join(d, f'TT_corr_matrix_l_2_{LMAX}_lp_2_{LMAX}.npy'); assert os.path.exists(src), src
        tmp = f + '.tmp'; shutil.copy(src, tmp)
        with open(tmp, 'rb') as fh: os.fsync(fh.fileno())
        os.replace(tmp, f)
    finally: os.chdir(cwd0)
    rec = dict(tag=tag, manifest=key_of(top, p, x0), cov_file_sha256=sha(f), cov_array_sha256=hashlib.sha256(np.load(f).tobytes()).hexdigest(), source_run_dir=os.path.basename(d.rstrip('/')), seconds=time.time() - t0)
    atomic_write_json(rec, mf); print(f'  generated {tag[:20]} ({time.time()-t0:.0f}s)', flush=True); return f, tag, rec
COV = {}; META = {}; t_all = time.time()
def get(top, p, x0):
    k = tag_of(top, p, x0)
    if k not in COV:
        f, tag, rec = ensure_cov(top, p, x0); Mx, Cr, meta = t1.load_cov_full(f, LMAX)
        assert Cr.shape == (21, 21); COV[k] = Cr; META[k] = dict(cache=rec, intake={kk: (vv if isinstance(vv, (int, float, str, bool, dict, list)) else str(vv)) for kk, vv in meta.items()})
    return COV[k]
for c in CASES:
    for x in ([c['base'], c['x0_H1'], c['x0_H2']] if A11_MODE == 'official' else [c['base'], c['x0_H1']]): get(c['topology'], c['shape'], x)
for c in INV: get(c['topology'], c['shape'], c['base']); get(c['topology'], c['shape'], c['x0'])
if A11_MODE == 'smoke':
    get('E2', p2, b1); f_, t_, r_ = ensure_cov('E2', p2, b1); r2 = ensure_cov('E2', p2, b1)[2]; GATES['G_smoke_cache_reuse'] = (r2['cov_file_sha256'] == r_['cov_file_sha256'])
GATES['G_unique_covariances'] = (len(COV) == EXPECTED['n_unique_cov'])
print(f'共分散 {len(COV)} 個（期待 {EXPECTED["n_unique_cov"]}）({(time.time()-t_all)/60:.1f} min)')

In [ ]:
# ---- 5. 橋渡し検証 ----
def rel(A, B): return float(np.linalg.norm(A - B) / np.linalg.norm(B))
def rel_off(A, B): o = lambda X: X - np.diag(np.diag(X)); return float(np.linalg.norm(o(A) - o(B)) / max(np.linalg.norm(o(B)), 1e-300))
def rel_white(A, B):
    d = np.diag(B); assert np.all(d > 0), 'white metric: non-positive diagonal'; s = 1 / np.sqrt(d); return float(np.linalg.norm((A - B) * np.outer(s, s)) / np.linalg.norm(B * np.outer(s, s)))
TOL_MATCH, TOL_DISCR = 1e-5, 1e-2; rows = []
for c in CASES:
    C0 = get(c['topology'], c['shape'], c['base']); target = D_of(c['M']) @ C0 @ D_of(c['M']).T; C1 = get(c['topology'], c['shape'], c['x0_H1'])
    C2 = get(c['topology'], c['shape'], c['x0_H2']) if A11_MODE == 'official' else None
    row = dict(case=c['case'], topology=c['topology'], gen=c['gen'], not_lattice_degenerate=c['not_lattice_degenerate'], rel_H1=rel(C1, target), off_H1=rel_off(C1, target), white_H1=rel_white(C1, target),
               rel_H2=(rel(C2, target) if C2 is not None else np.nan), off_H2=(rel_off(C2, target) if C2 is not None else np.nan), white_H2=(rel_white(C2, target) if C2 is not None else np.nan),
               rel_H1_vs_H2=(rel(C1, C2) if C2 is not None else np.nan), base_tag=tag_of(c['topology'], c['shape'], c['base'])[:20], H1_tag=tag_of(c['topology'], c['shape'], c['x0_H1'])[:20],
               H2_tag=(tag_of(c['topology'], c['shape'], c['x0_H2'])[:20] if C2 is not None else None))
    row['empirically_discriminating'] = (row['rel_H1_vs_H2'] > TOL_DISCR) if C2 is not None else None
    row['match_over_separation_H1'] = (row['rel_H1'] / max(row['rel_H1_vs_H2'], 1e-300)) if C2 is not None else np.nan
    rows.append(row)
for c in INV:
    C0 = get(c['topology'], c['shape'], c['base']); C1 = get(c['topology'], c['shape'], c['x0'])
    rows.append(dict(case=c['case'], topology=c['topology'], gen=c['kind'], not_lattice_degenerate=None, rel_H1=rel(C1, C0), off_H1=rel_off(C1, C0), white_H1=rel_white(C1, C0), rel_H2=np.nan, off_H2=np.nan, white_H2=np.nan,
                     rel_H1_vs_H2=np.nan, empirically_discriminating=None, match_over_separation_H1=np.nan, base_tag=tag_of(c['topology'], c['shape'], c['base'])[:20], H1_tag=tag_of(c['topology'], c['shape'], c['x0'])[:20], H2_tag=None))
df = pd.DataFrame(rows); df['H1_match'] = df.rel_H1 < TOL_MATCH; df['H2_match'] = df.rel_H2 < TOL_MATCH
disc = df[df.not_lattice_degenerate.eq(True)]; nond = df[df.not_lattice_degenerate.eq(False)]; lat = df[df.gen.str.endswith('lattice_translation')]
kern = df[df.gen.str.endswith(('invariant_plane_shift', 'kernel_shift'))]; ctrl = df[df.gen.eq('y_dependence_control')]
GATES['G_case_inventory'] = bool(set(disc.case) == EXPECTED['disc'] and set(nond.case) == EXPECTED['nond'] and set(lat.case) == EXPECTED['lat_cases'] and set(kern.case) == EXPECTED['kernel_cases'] and set(ctrl.case) == EXPECTED['ctrl'] and len(df) == EXPECTED['n_rows'])
assert GATES['G_case_inventory'], (set(disc.case), set(nond.case), set(lat.case), set(kern.case), set(ctrl.case), len(df))
if A11_MODE == 'official':
    H1m, H2m = disc.H1_match.eq(True).all(), disc.H2_match.eq(True).all(); H1f, H2f = (disc.rel_H1 > TOL_DISCR).all(), (disc.rel_H2 > TOL_DISCR).all(); sep = disc.empirically_discriminating.eq(True).all()
    G_H1 = bool(H1m and H2f and sep); G_H2 = bool(H2m and H1f and sep)
    CONVENTION = 'x0_CT = -r_obs (H1, registered canonical gauge)' if G_H1 else 'x0_CT = +r_obs (H2)' if G_H2 else 'UNRESOLVED'
    GATES['G_convention_H1'] = G_H1; GATES['G_E2_halfturn_M_part'] = bool(nond.H1_match.eq(True).all() and nond.H2_match.eq(True).all())
    for nm in ['E7', 'E7tilt', 'E8', 'E2']:
        sub = lat[lat.gen.eq(f'{nm}_lattice_translation')]; GATES[f'G_{nm}_lattice_translation_invariance_all_basis'] = bool(sub.H1_match.eq(True).all() and len(sub) == 3)
    for nm in ['E7_invariant_plane_shift', 'E7tilt_invariant_plane_shift', 'E8_kernel_shift', 'E2_kernel_shift']:
        sub = kern[kern.gen.eq(nm)]; GATES[f'G_{nm}'] = bool(sub.H1_match.eq(True).all() and len(sub) >= 1)
    GATES['G_y_dependence_control'] = bool((ctrl.rel_H1 > TOL_DISCR).all() and len(ctrl) == 1)
else:
    CONVENTION = 'smoke (H1 only): ' + ('H1 match' if bool(disc.H1_match.eq(True).all()) else 'H1 mismatch'); GATES['G_smoke_H1_match'] = bool(disc.H1_match.eq(True).all())
print(df[['case', 'gen', 'not_lattice_degenerate', 'rel_H1', 'rel_H2', 'rel_H1_vs_H2', 'white_H1', 'white_H2', 'H1_match', 'H2_match', 'empirically_discriminating']].to_string(index=False, float_format=lambda v: f'{v:.2e}'))
print('\n判定:', CONVENTION)

In [ ]:
# ---- 6. provenance（先に保存）→ exact gate inventory → 最終 assert ----
import platform
df.to_csv(os.path.join(OUT, 'a11_bridge_results.csv'), index=False)
COMMON = ['G_mt_commit', 'G_mt_origin', 'G_mt_clean_preimport', 'G_t1_engine_sha', 'G_bridge_sha', 'G_t2b2_run_sha', 'G_a6_artifact_sha', 'G_ct_commit', 'G_ct_origin', 'G_ct_clean_preimport_incl_untracked',
          'G_ct_dependencies_present', 'G_live_import_paths', 'G_live_module_sha', 'G_live_dependency_versions', 'G_quadrature_orthonormal', 'G_real_basis_bridge', 'G_D_orthogonal_all_used',
          'G_D_direct_geometry_complex_path', 'G_D_homomorphism', 'G_reflection_D_analytic', 'G_a6_ct_M_binding', 'G_a6_ct_T_binding', 'G_displacement_parallel_independent', 'G_displacement_perpendicular_independent',
          'G_displacement_components_excited', 'G_unique_covariances', 'G_case_inventory']
OFFICIAL_ONLY = ['G_env_lock', 'G_notebook_live_source', 'G_smoke_provenance_chain', 'G_convention_H1', 'G_E2_halfturn_M_part', 'G_E7_lattice_translation_invariance_all_basis', 'G_E7tilt_lattice_translation_invariance_all_basis',
                 'G_E8_lattice_translation_invariance_all_basis', 'G_E2_lattice_translation_invariance_all_basis', 'G_E7_invariant_plane_shift', 'G_E7tilt_invariant_plane_shift', 'G_E8_kernel_shift', 'G_E2_kernel_shift', 'G_y_dependence_control']
SMOKE_ONLY = ['G_env_lock_candidate_complete', 'G_notebook_head_available', 'G_smoke_cache_reuse', 'G_smoke_H1_match']
REQUIRED = COMMON + (OFFICIAL_ONLY if A11_MODE == 'official' else SMOKE_ONLY)
GATES = {k: bool(v) for k, v in GATES.items()}
GATES_EXACT = (set(GATES) == set(REQUIRED)); RUN_PASS = GATES_EXACT and all(GATES[k] for k in REQUIRED)
SMOKE_PASS = bool(RUN_PASS and A11_MODE == 'smoke'); OFFICIAL = bool(RUN_PASS and A11_MODE == 'official'); STATUS = 'SMOKE_PASS' if SMOKE_PASS else 'OFFICIAL' if OFFICIAL else 'FAILED'
prov = dict(notebook=f'Step1 Phase A-11 x0 bridge v1.3.1 [{A11_MODE}]', status=STATUS, SMOKE_PASS=SMOKE_PASS, OFFICIAL=OFFICIAL, notebook_identity=DIAG['notebook'], date=str(datetime.date.today()),
            timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(), conclusion=CONVENTION, gates=GATES, required_gates=REQUIRED, gate_inventory_exact=GATES_EXACT,
            tolerances=dict(match=TOL_MATCH, discriminate=TOL_DISCR), run_config=RUN_CFG,
            environment=dict(python=PYV, metadata_versions=VERS, live_versions=LIVE_VERS, requirements_sha256=REQ_SHA, platform=PLATFORM, blas_lapack=BLAS, fingerprint=ENV_FINGERPRINT,
                             full_pip_freeze_diagnostic_sha256=PIP_FREEZE_DIAG_SHA, lock_path=LOCK_PATH, lock_sha256=LOCK_SHA),
            repos=dict(mirror_topology=dict(commit=EXPECTED_MT_COMMIT, **mt), cmbtopology=dict(commit=EXPECTED_CMBTOPO_COMMIT, source_sha=CT_SRC_SHA, **ct)), frozen_sha=EXPECTED_SHA, git_calls=GIT_LOG, diagnostics=DIAG,
            cases=[{k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in c.items()} for c in CASES], invariance=[{k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in c.items()} for c in INV],
            expected_inventory={k: (sorted(v) if isinstance(v, set) else v) for k, v in EXPECTED.items()}, covariance_intake_meta=META, outputs=dict(results_csv_sha256=sha(os.path.join(OUT, 'a11_bridge_results.csv'))))
PROV_PATH = os.path.join(OUT, 'a11_provenance.json'); atomic_write_json(prov, PROV_PATH)
if SMOKE_PASS:                                           # the environment lock is written ONLY by a successful smoke run
    LOCK_CANDIDATE.update(smoke_status='SMOKE_PASS', smoke_provenance_path=PROV_PATH, smoke_provenance_sha256=sha(PROV_PATH), canonical_notebook_head_sha=NB_HEAD_SHA,
                          generated=datetime.datetime.now(datetime.timezone.utc).isoformat())
    atomic_write_json(LOCK_CANDIDATE, LOCK_PATH); print('[smoke] environment lock written:', LOCK_PATH)
print('STATUS =', STATUS, '/ 判定:', CONVENTION, '/ saved:', OUT)
failed = {k: v for k, v in GATES.items() if v is not True}; missing = set(REQUIRED) - set(GATES); extra = set(GATES) - set(REQUIRED)
assert RUN_PASS, f'A11 run FAILED [{A11_MODE}]: failed={failed} missing={missing} unregistered={extra}'
if A11_MODE == 'official':
    assert OFFICIAL and CONVENTION == 'x0_CT = -r_obs (H1, registered canonical gauge)', CONVENTION
else:
    assert SMOKE_PASS

## 実行手順
1. **このノートブック（v1.3.1）をリポジトリ直下に commit・push**（notebook 同一性 gate が origin/main の同名ファイルと比較するため）。
2. **smoke**：先頭に `A11_MODE = 'smoke'` のセルを追加した**コピー**で実行（約 20 分・3 共分散）。`runs_step1_phaseA/a11_env_lock.json` が生成される。
3. **official**：純正の v1.3.1（セル追加なし）を Runtime restart → Run all（約 4 時間・35 共分散・cache 再開可）。最後に `assert OFFICIAL`。
4. `a11_v1.3.1_smoke/`・`a11_v1.3.1_official/` の `a11_bridge_results.csv`・`a11_provenance.json`（＋`cov_cache/`）と全セル出力を返送。